In [ ]:
from google.colab import auth
auth.authenticate_user()

In [1]:
import pandas as pd
from datetime import datetime
import time
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import zipfile
import os
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
pd.set_option('display.max_columns', None)
from google.cloud import bigquery
from google.api_core.exceptions import NotFound, GoogleAPICallError

In [ ]:
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
BUCKET_NAME = "rs-nprd-dlk-ue4-gcs-ryl-sftp_generics"
FOLDER_PATH= "data_entries/SUSTENTO_TRANSFORMADO/"
DATASET_ID = "produccion"
TABLE_ID= "TRAMAS_SUSTENTO_prestamos"
TABLE_CONTROL_ID= "CONTROL_tramas_sustento"


In [ ]:
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

letra = np.array(list("ABCDEFGHIJKLMNOPQRSTUVWXYZ{0123456789"))
valor = np.array(list("1234567891234567890000000000123456789"))
diccionario_reemplazo = dict(zip(letra, valor))

espacios_col = [(0, 4), (4, 34), (34, 37), (37, 57), (57, 60), 
                (60, 70), (70, 90), (90, 110), (110, 111), (111, 121), 
                (121, 131), (131, 146), (146, 161), (161, 176), (176, 178),
                (178, 196), (196, 211), (211, 221), (221, 223), (223, 225), 
                (225, 232), (232, 240), (240, 248), (248, 256), (256, 258),
                (258, 273), (273, 275), (275, 290), (290, 305), (305, 315), (315, 325)]

column_names = ['Oficina','Descripcion_Oficina','Codigo_Subproducto','Descripcion_Producto','Moneda',
                'Nro_Poliza','Nro_Certificado','Nro_Cuenta','Forma_de_Pago','Fecha_de_Cobro',
                'Fecha_de_Liquidacion','Prima','Comision','%_Comision','Indicador_de_Renovacion',
                'Filler','Monto_Asegurado','Tasa_sin_Recargo','Plan_de_credito','Planes_Tipo_Desgravamen',
                '%_recargo','Fec_Inicio_Pago','Fec_Fin_Pago','Fecha_Inicio_de_Vigencia_Poliza', 'Modalidad', 
                'Prima_Rimac','Compañia','Prima_Desempleo','Prima_Desgravamen', 'Fecha_Inicio_V2', 'Fecha_Fin_V2']

schema_Trama = [
        bigquery.SchemaField("OFICINA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DESCRIPCION_OFICINA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_SUBPRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DESCRIPCION_PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NRO_POLIZA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NRO_CERTIFICADO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NRO_CUENTA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FORMA_DE_PAGO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_DE_COBRO", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_DE_LIQUIDACION", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("PRIMA ", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("COMISION", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("%_COMISION", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("INDICADOR_DE_RENOVACION", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FILLER", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MONTO_ASEGURADO", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("TASA_SIN_RECARGO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PLAN_DE_CREDITO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PLANES_TIPO_DESGRAVAMEN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("%_RECARGO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FEC_INICIO_PAGO", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FEC_FIN_PAGO ", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_INICIO_DE_VIGENCIA_POLIZA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("MODALIDAD", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA_RIMAC ", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("COMPAÑIA ", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA_DESEMPLEO ", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("PRIMA_DESGRAVAMEN ", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("FECHA_INICIO_V2 ", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_FIN_V2 ", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("TRAMA_ORIGINAL ", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_RECEPCION ", bigquery.enums.SqlTypeNames.DATE)    
]

schema_Control = [
        bigquery.SchemaField("ARCHIVO_TXT", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.DATETIME)
]



# Extraer Fecha del nombre de archivo
def Extraer_fechas(filename):
    try:
        decl_str = filename[4:8]  # "1025"
        dia= 1 
        mes = int(decl_str[:2])
        year= int('20'+ decl_str[2:])
        fecha = datetime(year, mes, dia).date()
        return fecha
    except Exception:
        return None # Si algo falla


# Convertir fechas string a tipo datetime
def Convertir_fecha(columna):
    col = columna.astype(str) # Convertir a string
    col = col.str.strip()  # Eliminar espacios
    # Quitar decimales tipo '19880329.0'
    col = col.str.replace(r"\.0$", "", regex=True)
    col = col.str[:8] # Mantener solo los 8 primeros caracteres
    # Reemplazar valores inválidos (que no tengan 8 dígitos) por NaN
    col = col.where(col.str.match(r"^\d{8}$"), np.nan)
    return pd.to_datetime(col, format="%Y%m%d", errors="coerce")  # Convertir a fecha


# Convertir cada línea en un dict según los cortes
def Convertir_linea(linea):
    row= {name: linea[start:end].strip() for (start, end), name in zip(espacios_col, column_names)}
    row["Trama Original"] = linea.rstrip("\n")
    return row


def convertir_prima(valor_str):
    try:
        # Debe ser al menos 2 caracteres (números + letra)
        if not isinstance(valor_str, str) or len(valor_str) < 2:
            return np.nan

        parte_numerica = valor_str[:-1]
        letra_final = valor_str[-1]
        # Buscar equivalencia
        digito = diccionario_reemplazo.get(letra_final)
        if digito is None:
            return np.nan  # letra desconocida → nulo
        # Construir número completo
        numero_str = parte_numerica + digito
        # Intentar convertir a Decimal
        return float(numero_str) / 100

    except Exception:
        return np.nan  # En caso de cualquier error, devolver nulo

def tabla_existe(table_ref):
    try:
      tabla = bigquery_client.get_table(table_ref)
      return True
    except NotFound:
        # La tabla no existe
        return False
    except GoogleAPICallError as e:
        # Otros errores de BigQuery (permisos, conexión, etc.)
        print(f"⚠️ Error al consultar BigQuery: {e}")
        raise


def buscar_archivo(filename_txt: str) -> bool:
    # Verificar si existe la tabla de control
    table_control_ref = bigquery_client.dataset(DATASET_ID).table(TABLE_CONTROL_ID)
    if not tabla_existe(table_control_ref):
        return False
    else:
        # Buscar archivo en la tabla de control
        query = f"""
            SELECT COUNT(*) AS count
            FROM `{table_control_ref}`
            WHERE ARCHIVO_TXT = @archivo_txt
        """
        job_config = bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("archivo_txt", "STRING", filename_txt)]
        )

        df = bigquery_client.query(query, job_config=job_config).to_dataframe()
        return df["count"].iloc[0] > 0


#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    job_config = bigquery.LoadJobConfig()
    if tabla_existe(table_ref):
        # Abre la tabla para agregar registros
        job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND
    else:
        tabla_tramas = bigquery.Table(table_ref, schema=schema)
        tabla_tramas = bigquery_client.create_table(tabla_tramas)
        job_config.write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE
        print(f'ℹ️ ----- Se ha creado la tabla: {table_id} en el dataset: {dataset_id} -----')
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    #print(f'----- REGISTROS AGREGADOS CORRECTAMENTE EN: {table_id} -------')
    return


def Procesar_File_txt(lista_archivos):
    for blob in lista_archivos:
        if blob.name.endswith(".TXT"):
            file_name = os.path.basename(blob.name)
            if not buscar_archivo(file_name):
                print(f"⏳ ... PROCESANDO ARCHIVO: {blob.name} ... ⏳")
                file_bytes = blob.download_as_bytes()
                try:
                    contenido = file_bytes.decode('latin-1')
                except UnicodeDecodeError:
                    # Si falla, usar UTF-8
                    contenido = file_bytes.decode('utf-8', errors='replace')
                    
                lista_lineas = contenido.splitlines()
                df_tramas.columns = (df_tramas.columns.str.upper())  # opcional: todo en mayúsculas
            
                df_tramas = pd.DataFrame([Convertir_linea(linea) for linea in lista_lineas])
                df_tramas['FECHA_DE_COBRO'] = pd.to_datetime(df_tramas['FECHA_DE_COBRO'],format='%Y-%m-%d', errors='coerce').dt.date
                df_tramas['FECHA_DE_LIQUIDACION'] = pd.to_datetime(df_tramas['FECHA_DE_LIQUIDACION'],format='%Y-%m-%d', errors='coerce').dt.date
                df_tramas['FEC_INICIO_PAGO'] = pd.to_datetime(df_tramas['FEC_INICIO_PAGO'],format='%Y%m%d', errors='coerce').dt.date
                df_tramas['FEC_FIN_PAGO'] = pd.to_datetime(df_tramas['FEC_FIN_PAGO'],format='%Y%m%d', errors='coerce').dt.date
                df_tramas['FECHA_INICIO_DE_VIGENCIA_POLIZA'] = pd.to_datetime(df_tramas['FECHA_INICIO_DE_VIGENCIA_POLIZA'],format='%d%m%Y', errors='coerce').dt.date
                df_tramas['FECHA_INICIO_V2'] = pd.to_datetime(df_tramas['FECHA_INICIO_V2'],format='%Y-%m-%d', errors='coerce').dt.date
                df_tramas['FECHA_FIN_V2'] = pd.to_datetime(df_tramas['FECHA_FIN_V2'],format='%Y-%m-%d', errors='coerce').dt.date

                df_tramas["PRIMA"] = df_tramas["PRIMA"].apply(convertir_prima)
                df_tramas["COMISION"] = df_tramas["COMISION"].apply(convertir_prima)
                df_tramas["MONTO_ASEGURADO"] = df_tramas["MONTO_ASEGURADO"].apply(convertir_prima)
                df_tramas["PRIMA_RIMAC"] = df_tramas["PRIMA_RIMAC"].apply(convertir_prima)
                df_tramas["PRIMA_DESEMPLEO"] = df_tramas["PRIMA_DESEMPLEO"].apply(convertir_prima)
                df_tramas["PRIMA_DESGRAVAMEN"] = df_tramas["PRIMA_DESGRAVAMEN"].apply(convertir_prima)
                
                df_tramas["NOMBRE_ARCHIVO"] = file_name
                df_tramas['FECHA_RECEPCION'] = Extraer_fechas(file_name)
            
                # Guardar tabla en BigQuery
                fecha_carga = datetime.now()
                df_control = pd.DataFrame([{"ARCHIVO_TXT": file_name, "FECHA_CARGA": fecha_carga}])
                Guardar_en_BigQuery(df_tramas, DATASET_ID, TABLE_ID, schema_Trama)
                Guardar_en_BigQuery(df_control, DATASET_ID, TABLE_CONTROL_ID, schema_Control)
                print(f"✅ ARCHIVO {file_name} CARGADO ...")
            else:
                print(f"⚠️ EL ARCHIVO {file_name} YA FUE CARGADO ANTERIORMENTE ...")
    return

##################### PRINCIPAL #########################
bucket = storage_client.bucket(BUCKET_NAME)
blob_list = list(bucket.list_blobs(prefix=FOLDER_PATH))
Procesar_File_txt(blob_list)


⏳ ... PROCESANDO ARCHIVO: data_entries/SUSTENTO_TRANSFORMADO/TRAMAS_SUSTENTO_PRESTAMOS.TXT ... ⏳
ℹ️ ----- Se ha creado la tabla: TRAMAS_SUSTENTO_prestamos en el dataset: produccion -----
ℹ️ ----- Se ha creado la tabla: CONTROL_tramas_sustento en el dataset: produccion -----
✅ ARCHIVO TRAMAS_SUSTENTO_PRESTAMOS.TXT CARGADO ...


----

In [ ]:
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

In [ ]:
bucket = storage_client.bucket(BUCKET_NAME)
blob_list = list(bucket.list_blobs(prefix=FOLDER_PATH))
for blob in blob_list:
    if blob.name.endswith(".TXT"):
      print(f"⏳ ... PROCESANDO ARCHIVO: {blob.name} ... ⏳")

⏳ ... PROCESANDO ARCHIVO: data_cpozo/tramas_DTC/2019/Octubre/RED011019_0210.TXT ... ⏳
⏳ ... PROCESANDO ARCHIVO: data_cpozo/tramas_DTC/2019/Octubre/RED021019_0310.TXT ... ⏳
⏳ ... PROCESANDO ARCHIVO: data_cpozo/tramas_DTC/2019/Octubre/RED031019_0410.TXT ... ⏳
⏳ ... PROCESANDO ARCHIVO: data_cpozo/tramas_DTC/2019/Octubre/RED041019_0710.TXT ... ⏳
⏳ ... PROCESANDO ARCHIVO: data_cpozo/tramas_DTC/2019/Octubre/RED101019_1110.TXT ... ⏳
⏳ ... PROCESANDO ARCHIVO: data_cpozo/tramas_DTC/2019/Octubre/RED111019_1410.TXT ... ⏳
⏳ ... PROCESANDO ARCHIVO: data_cpozo/tramas_DTC/2019/Octubre/RED141019_1510.TXT ... ⏳
⏳ ... PROCESANDO ARCHIVO: data_cpozo/tramas_DTC/2019/Octubre/RED151019_1610.TXT ... ⏳
⏳ ... PROCESANDO ARCHIVO: data_cpozo/tramas_DTC/2019/Octubre/RED161019_1710.TXT ... ⏳
⏳ ... PROCESANDO ARCHIVO: data_cpozo/tramas_DTC/2019/Octubre/RED171019_1810.TXT ... ⏳
⏳ ... PROCESANDO ARCHIVO: data_cpozo/tramas_DTC/2019/Octubre/RED181019_2110.TXT ... ⏳
⏳ ... PROCESANDO ARCHIVO: data_cpozo/tramas_DTC/2019/O

In [ ]:
print(blob_list[1].name)
file_name = os.path.basename(blob_list[1].name)
print(file_name)
file_bytes = blob_list[1].download_as_bytes()
try:
  contenido = file_bytes.decode('latin-1')
except UnicodeDecodeError:
      # Si falla, usar UTF-8
  contenido = file_bytes.decode('utf-8', errors='replace')
lista_lineas = contenido.splitlines()
df_tramas = pd.DataFrame([Convertir_linea(linea) for linea in lista_lineas])
df_tramas.head(3)

data_cpozo/tramas_DTC/2018/Agosto/RED010818_0208.TXT
RED010818_0208.TXT


,Tipo de seguro,Certificado,Numero Interno Del Canal,Tipo de Registro,Moneda,Tipo de Movimiento,Fecha de Afiliacion,Fecha de inicio del seguro,Fecha fin del seguro,Periodo de pago,Plan de Seguro,Prima,Trama Original
0,703,00110003164000009840,0183,00,PEN,2,20180801,20150224,20180825,M,,00000000000480I,703001100031640000098400183 00P...
1,703,00110030424000035539,0030,00,PEN,4,20180801,20180724,20180824,M,,00000000000480I,703001100304240000355390030 00P...
2,703,00110102144000280577,0102,00,PEN,4,20180801,20180731,20180831,M,,00000000000480I,703001101021440002805770102 00P...


In [2]:
letra = np.array(list("ABCDEFGHIJKLMNOPQRSTUVWXYZ{0123456789"))
valor = np.array(list("1234567891234567890000000000123456789"))
diccionario_reemplazo = dict(zip(letra, valor))

espacios_col = [(0, 4), (4, 34), (34, 37), (37, 57), (57, 60), 
                (60, 70), (70, 90), (90, 110), (110, 111), (111, 121), 
                (121, 131), (131, 146), (146, 161), (161, 176), (176, 178),(178, 196),(196, 211),
                (211, 221), (221, 223), (223, 225), (225, 232), (232, 240), (240, 248), (248, 256), (256, 258),
                (258, 273), (273, 275), (275, 290), (290, 305), (305, 315), (315, 325)]

column_names = ['Oficina','Descripcion_Oficina','Codigo_Subproducto','Descripcion_Producto','Moneda',
                'Nro_Poliza','Nro_Certificado','Nro_Cuenta','Forma_de_Pago','Fecha_de_Cobro',
                'Fecha_de_Liquidacion','Prima','Comision','%_Comision','Indicador_de_Renovacion',
                'Filler','Monto_Asegurado','Tasa_sin_Recargo','Plan_de_credito','Planes_Tipo_Desgravamen',
                '%_recargo','Fec_Inicio_Pago','Fec_Fin_Pago','Fecha_Inicio_de_Vigencia_Poliza','Modalidad',
                'Prima_Rimac','Compañia','Prima_Desempleo','Prima_Desgravamen', 'Fecha_Inicio_V2', 'Fecha_Fin_V2']


# Extraer Fecha del nombre de archivo
def Extraer_fechas(filename):
    try:
        decl_str = filename[4:8]  # "1025"
        dia= 1 
        mes = int(decl_str[:2])
        year= int('20'+ decl_str[2:])
        fecha = datetime(year, mes, dia).date()
        return fecha
    except Exception:
        return None # Si algo falla


# Convertir fechas string a tipo datetime
def Convertir_fecha(columna):
    col = columna.astype(str) # Convertir a string
    col = col.str.strip()  # Eliminar espacios
    # Quitar decimales tipo '19880329.0'
    col = col.str.replace(r"\.0$", "", regex=True)
    col = col.str[:8] # Mantener solo los 8 primeros caracteres
    # Reemplazar valores inválidos (que no tengan 8 dígitos) por NaN
    col = col.where(col.str.match(r"^\d{8}$"), np.nan)
    return pd.to_datetime(col, format="%Y%m%d", errors="coerce")  # Convertir a fecha


# Convertir cada línea en un dict según los cortes
def Convertir_linea(linea):
    row= {name: linea[start:end].strip() for (start, end), name in zip(espacios_col, column_names)}
    row["Trama_Original"] = linea.rstrip("\n")
    return row


def convertir_prima(valor_str):
    try:
        # Debe ser al menos 2 caracteres (números + letra)
        if not isinstance(valor_str, str) or len(valor_str) < 2:
            return np.nan

        parte_numerica = valor_str[:-1]
        letra_final = valor_str[-1]
        # Buscar equivalencia
        digito = diccionario_reemplazo.get(letra_final)
        if digito is None:
            return np.nan  # letra desconocida → nulo
        # Construir número completo
        numero_str = parte_numerica + digito
        # Intentar convertir a Decimal
        return float(numero_str) / 100

    except Exception:
        return np.nan  # En caso de cualquier error, devolver nulo


In [12]:
with open("C:/data/SUST1025(V2).TXT", "r", encoding="latin-1") as file:
    lista_lineas =  file.readlines()

In [13]:
df_tramas = pd.DataFrame([Convertir_linea(linea) for linea in lista_lineas])
df_tramas.columns = (df_tramas.columns.str.upper())  # opcional: todo en mayúsculas

In [15]:
df_tramas.shape

(1210292, 32)

In [6]:
df_tramas['FECHA_DE_COBRO'] = pd.to_datetime(df_tramas['FECHA_DE_COBRO'],format='%Y-%m-%d', errors='coerce').dt.date
df_tramas['FECHA_DE_LIQUIDACION'] = pd.to_datetime(df_tramas['FECHA_DE_LIQUIDACION'],format='%Y-%m-%d', errors='coerce').dt.date
df_tramas['FEC_INICIO_PAGO'] = pd.to_datetime(df_tramas['FEC_INICIO_PAGO'],format='%Y%m%d', errors='coerce').dt.date
df_tramas['FEC_FIN_PAGO'] = pd.to_datetime(df_tramas['FEC_FIN_PAGO'],format='%Y%m%d', errors='coerce').dt.date
df_tramas['FECHA_INICIO_DE_VIGENCIA_POLIZA'] = pd.to_datetime(df_tramas['FECHA_INICIO_DE_VIGENCIA_POLIZA'],format='%d%m%Y', errors='coerce').dt.date
df_tramas['FECHA_INICIO_V2'] = pd.to_datetime(df_tramas['FECHA_INICIO_V2'],format='%Y-%m-%d', errors='coerce').dt.date
df_tramas['FECHA_FIN_V2'] = pd.to_datetime(df_tramas['FECHA_FIN_V2'],format='%Y-%m-%d', errors='coerce').dt.date

In [7]:
df_tramas["PRIMA"] = df_tramas["PRIMA"].apply(convertir_prima)
df_tramas["COMISION"] = df_tramas["COMISION"].apply(convertir_prima)
df_tramas["MONTO_ASEGURADO"] = df_tramas["MONTO_ASEGURADO"].apply(convertir_prima)
df_tramas["PRIMA_RIMAC"] = df_tramas["PRIMA_RIMAC"].apply(convertir_prima)
df_tramas["PRIMA_DESEMPLEO"] = df_tramas["PRIMA_DESEMPLEO"].apply(convertir_prima)
df_tramas["PRIMA_DESGRAVAMEN"] = df_tramas["PRIMA_DESGRAVAMEN"].apply(convertir_prima)

In [8]:
df_tramas['FECHA_RECEPCION'] = Extraer_fechas('SUST1225(V2).TXT')

In [9]:
df_tramas.head()

,OFICINA,DESCRIPCION_OFICINA,CODIGO_SUBPRODUCTO,DESCRIPCION_PRODUCTO,MONEDA,NRO_POLIZA,NRO_CERTIFICADO,NRO_CUENTA,FORMA_DE_PAGO,FECHA_DE_COBRO,FECHA_DE_LIQUIDACION,PRIMA,COMISION,%_COMISION,INDICADOR_DE_RENOVACION,FILLER,MONTO_ASEGURADO,TASA_SIN_RECARGO,PLAN_DE_CREDITO,PLANES_TIPO_DESGRAVAMEN,%_RECARGO,FEC_INICIO_PAGO,FEC_FIN_PAGO,FECHA_INICIO_DE_VIGENCIA_POLIZA,MODALIDAD,PRIMA_RIMAC,COMPAÑIA,PRIMA_DESEMPLEO,PRIMA_DESGRAVAMEN,FECHA_INICIO_V2,FECHA_FIN_V2,TRAMA_ORIGINAL,FECHA_RECEPCION
0,0005,OFICINA MEGAPLAZA CHIMBOTE,952,SEG.DESG.TRIANG.VEH,PEN,5000001,00110005694000356584,00110814150286287323,M,NaT,2025-12-26,24.87,12.79,05295000,,        ,34883.1,0006900000,01,01,0000000,2025-11-28,2025-12-29,2025-11-28,02,0.0,01,0.0,0.0,2025-11-28,2025-12-29,0005OFICINA MEGAPLAZA CHIMBOTE 952SEG.DESG....,2025-12-01
1,0010,OFICINA ANDAHUAYLAS,901,SEG.DESG.CONTIFAC.,PEN,5143910,00110831584001241366,00110010670200283796,M,2025-12-10,2025-12-01,8.12,4.16,05278000,,        ,NaN,          ,  ,  ,       ,NaT,NaT,2022-01-14,02,0.0,01,0.0,0.0,2025-10-30,2025-12-01,0010OFICINA ANDAHUAYLAS 901SEG.DESG....,2025-12-01
2,0010,OFICINA ANDAHUAYLAS,901,SEG.DESG.CONTIFAC.,PEN,5143910,00110814174002778026,00110010650200071314,M,2025-12-01,2025-12-01,8.67,4.44,05278000,,        ,NaN,          ,  ,  ,       ,NaT,NaT,2022-07-18,02,0.0,01,0.0,0.0,2025-10-30,2025-12-01,0010OFICINA ANDAHUAYLAS 901SEG.DESG....,2025-12-01
3,0010,OFICINA ANDAHUAYLAS,901,SEG.DESG.CONTIFAC.,PEN,5143910,00110010674000416517,00110368850200479515,M,2025-12-01,2025-12-01,32.78,16.80,05278000,,        ,NaN,          ,  ,  ,       ,NaT,NaT,2021-10-26,02,0.0,01,0.0,0.0,2025-10-30,2025-12-01,0010OFICINA ANDAHUAYLAS 901SEG.DESG....,2025-12-01
4,0010,OFICINA ANDAHUAYLAS,901,SEG.DESG.CONTIFAC.,PEN,5143910,00110010684000416282,00110814160214312889,M,NaT,2025-12-01,4.66,2.39,05278000,,        ,NaN,          ,  ,  ,       ,NaT,NaT,2021-10-25,02,0.0,01,0.0,0.0,2025-10-30,2025-12-01,0010OFICINA ANDAHUAYLAS 901SEG.DESG....,2025-12-01


In [10]:
df_tramas['CODIGO_SUBPRODUCTO'].value_counts()

CODIGO_SUBPRODUCTO
951    262035
918    195353
958     52605
901     35409
903     33914
961     21890
952     16347
953     15671
972      5752
906      5619
968      3264
919      3232
954      3009
902      2709
962      2608
963      1486
964       885
966       489
941       477
959       106
Name: count, dtype: int64

In [59]:
df_tramas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 662860 entries, 0 to 662859
Data columns (total 33 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   OFICINA                          662860 non-null  object 
 1   DESCRIPCION_OFICINA              662860 non-null  object 
 2   CODIGO_SUBPRODUCTO               662860 non-null  object 
 3   DESCRIPCION_PRODUCTO             662860 non-null  object 
 4   MONEDA                           662860 non-null  object 
 5   NRO_POLIZA                       662860 non-null  object 
 6   NRO_CERTIFICADO                  662860 non-null  object 
 7   NRO_CUENTA                       662860 non-null  object 
 8   FORMA_DE_PAGO                    662860 non-null  object 
 9   FECHA_DE_COBRO                   577510 non-null  object 
 10  FECHA_DE_LIQUIDACION             662860 non-null  object 
 11  PRIMA                            662860 non-null  float64
 12  CO

In [ ]:
DATASET_ID = "produccion"
TABLE_CONTROL_ID= "CONTROL_tramas"
table_control_ref = bigquery_client.dataset(DATASET_ID).table(TABLE_CONTROL_ID)

schema_TABLA_C = [
        bigquery.SchemaField("ARCHIVO_TXT", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.DATETIME)
]

In [ ]:
schema_Trama = [
        bigquery.SchemaField("OFICINA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DESCRIPCION_OFICINA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_SUBPRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DESCRIPCION_PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NRO_POLIZA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NRO_CERTIFICADO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NRO_CUENTA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FORMA_DE_PAGO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_DE_COBRO", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_DE_LIQUIDACION", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("PRIMA ", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("COMISION", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("%_COMISION", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("INDICADOR_DE_RENOVACION", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FILLER", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MONTO_ASEGURADO", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("TASA_SIN_RECARGO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PLAN_DE_CREDITO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PLANES_TIPO_DESGRAVAMEN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("%_RECARGO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FEC_INICIO_PAGO", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FEC_FIN_PAGO ", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_INICIO_DE_VIGENCIA_POLIZA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("MODALIDAD", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA_RIMAC ", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("COMPAÑIA ", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA_DESEMPLEO ", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("PRIMA_DESGRAVAMEN ", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("FECHA_INICIO_V2 ", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_FIN_V2 ", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("TRAMA_ORIGINAL ", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_RECEPCION ", bigquery.enums.SqlTypeNames.DATE)    
]